# NumPy Foundations - Session 1
## The Backbone of Data Science & Machine Learning

---

### What You'll Learn in This Session

By the end of this session, you will be able to:

1. **Understand WHY** NumPy is essential (not just WHAT it does)
2. **Create** arrays from scratch using multiple methods
3. **Access** data using indexing and slicing
4. **Manipulate** array shapes and combine arrays
5. **Avoid common bugs** related to views vs. copies

---

### Why Should I Care About NumPy?

| Your Goal | How NumPy Helps |
|-----------|----------------|
| **Data Analysis** | Load CSVs → Pandas uses NumPy under the hood |
| **Machine Learning** | Every model (scikit-learn, TensorFlow, PyTorch) expects NumPy arrays |
| **Image Processing** | Images are just 3D arrays (height x width x RGB) |
| **Financial Analysis** | Stock prices, returns, correlations = array operations |
---


> **Real Talk**: If Python lists were bicycles, NumPy arrays are race cars. Same roads, 100x faster.

---

## Chapter 1: The "Why" — NumPy's Performance Advantage

### Why Not Just Use Python Lists?

To understand NumPy's value, we need to peek under Python's hood.

### 1.1 Python's Flexibility Tax

#### A Python Integer is NOT Just a Number

In C, an integer is literally 4 bytes of memory storing a number. In Python 3, a single integer contains **four pieces**:

| Field | Purpose |
|-------|--------|
| `ob_refcnt` | Reference count for memory management |
| `ob_type` | Encodes the variable type |
| `ob_size` | Specifies size of data members |
| `ob_digit` | The actual integer value |

**This means**: A single Python `int` uses ~28 bytes vs. C's 4 bytes!

#### Visual Comparison: C Integer vs. Python Integer

```
C Integer (4 bytes)          Python Integer (~28 bytes)
┌──────────────┐             ┌──────────────────────────┐
│    value     │             │ PyObject_HEAD            │
│     (4)      │             │  ├── ob_refcnt (8)       │
└──────────────┘             │  ├── ob_type (8)         │
                             │  └── ob_size (8)         │
                             │ ob_digit (4+)            │
                             └──────────────────────────┘
```



### 1.2 Python List vs. NumPy Array: Memory Layout

#### Python List: Scattered Memory
```
Python List [1, 2, 3, 4]
┌──────────────────────────────────────────────────┐
│ List Object                                      │
│  ├── pointer → PyInt(1) at address 0x1A...      │
│  ├── pointer → PyInt(2) at address 0x2B...      │
│  ├── pointer → PyInt(3) at address 0x3C...      │
│  └── pointer → PyInt(4) at address 0x4D...      │
└──────────────────────────────────────────────────┘
```
Each access requires: **Dereference pointer → Find object → Extract value**

#### NumPy Array: Contiguous Memory
```
NumPy Array [1, 2, 3, 4] (dtype=int64)
┌────┬────┬────┬────┐
│ 1  │ 2  │ 3  │ 4  │  ← Values sit side-by-side in memory
└────┴────┴────┴────┘
```
Each access: **Jump directly to memory location**

---

### Real-World Impact: Why This Matters

| Scenario | Python List Time | NumPy Time | Speedup |
|----------|-----------------|------------|--------|
| Sum 1 million numbers | ~50ms | ~0.5ms | **100x** |
| Multiply arrays | ~200ms | ~2ms | **100x** |
| Load 10GB dataset | Out of memory | Works fine | ∞ |

In [ ]:
# Let's prove it! Run this cell to see the difference
import numpy as np
import time

# Create test data
py_list = list(range(1_000_000))
np_array = np.array(py_list)

# Time Python list sum
start = time.time()
sum(py_list)
py_time = time.time() - start

# Time NumPy sum
start = time.time()
np.sum(np_array)
np_time = time.time() - start

print(f"Python list sum: {py_time*1000:.2f} ms")
print(f"NumPy array sum: {np_time*1000:.2f} ms")
print(f"NumPy is {py_time/np_time:.0f}x faster!")

Python list sum: 43.01 ms
NumPy array sum: 4.75 ms
NumPy is 9x faster!


---

## Chapter 2: Creating NumPy Arrays

### When Will You Use This?

| Creation Method | Real-World Use Case |
|-----------------|--------------------|
| `np.array()` | Converting existing data (CSV, lists, API responses) |
| `np.zeros()` | Initializing weight matrices in neural networks |
| `np.ones()` | Creating bias vectors, masks |
| `np.arange()` | Generating time series indices, batch IDs |
| `np.linspace()` | Creating smooth curves for visualization |
| `np.random.*` | Initializing model weights, Monte Carlo simulations |

### 2.1 From Python Lists → NumPy Arrays

In [ ]:
import numpy as np  # Standard convention: always import as 'np'

# Simple 1D array
arr = np.array([1, 4, 2, 5, 3])
print(f"1D Array: {arr}")
print(f"Type: {type(arr)}")

1D Array: [1 4 2 5 3]
Type: <class 'numpy.ndarray'>


### Important: Type Upcasting

NumPy arrays can only contain **one data type**. If types don't match, NumPy will **upcast** to the most general type.

In [ ]:
# Mixed types: int + float → all become float
mixed = np.array([3.14, 4, 2, 3])
print(f"Array: {mixed}")
print(f"Element mixed[1] = {mixed[1]}, type = {type(mixed[1])}")

# The integer 4 became 4.0!

Array: [3.14 4.   2.   3.  ]
Element mixed[1] = 4.0, type = <class 'numpy.float64'>


In [ ]:
# Mixed types: int + float → all become float
mixed = np.array([3.14, 4, 2, 3,"kero"])
print(f"Array: {mixed}")
print(f"Element mixed[1] = {mixed[1]}, type = {type(mixed[1])}")

# The integer 4 became 4.0!

Array: ['3.14' '4' '2' '3' 'kero']
Element mixed[1] = 4, type = <class 'numpy.str_'>


### 2.2 Specifying Data Types with `dtype`

**Why care about dtype?**
- `float32` uses half the memory of `float64` → Train larger models
- `int8` for image pixels (0-255) → Faster processing
- Wrong dtype = wrong results (e.g., integer overflow)

In [ ]:
# Explicit dtype specification
arr_float32 = np.array([1, 2, 3, 4], dtype=np.float32)
arr_int8 = np.array([1, 2, 3, 4], dtype=np.int8)

print(f"float32: {arr_float32}, uses {arr_float32.nbytes} bytes")
print(f"int8:    {arr_int8}, uses {arr_int8.nbytes} bytes")

float32: [1. 2. 3. 4.], uses 16 bytes
int8:    [1 2 3 4], uses 4 bytes


### 2.3 Creating Multidimensional Arrays

**Mental Model**: Think of dimensions as nested boxes
- 1D = A line of boxes (vector)
- 2D = A grid of boxes (matrix, like Excel)
- 3D = A cube of boxes (like RGB image: height x width x 3 channels)

In [ ]:
# Python nested list → 2D NumPy array
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])
print("2D Matrix:")
print(matrix)
print(f"\nShape: {matrix.shape}  # (rows, columns)")

2D Matrix:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Shape: (3, 3)  # (rows, columns)


### 2.4 Creating Arrays from Scratch

These functions are your bread and butter for initializing data structures.

In [ ]:
# 1. Zeros: Initialize neural network weights, create empty masks
zeros = np.zeros(10)
print(f"Zeros (1D): {zeros}")

zeros_2d = np.zeros((3, 5))
print(f"\nZeros (3x5 matrix):")
print(zeros_2d)

Zeros (1D): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Zeros (3×5 matrix):
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [ ]:
# 2. Ones: Bias initialization, creating masks
ones = np.ones((3, 5), dtype='float')
print(f"Ones (3x5, float):")
print(ones)

Ones (3×5, float):
[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]


In [ ]:
# 3. Full: Initialize with any constant
# Use case: Set all predictions to 0.5 initially
full = np.full((3, 5), 0.5)
print(f"Full (3x5, value=0.5):")
print(full)

Full (3x5, value=0.5):
[[0.5 0.5 0.5 0.5 0.5]
 [0.5 0.5 0.5 0.5 0.5]
 [0.5 0.5 0.5 0.5 0.5]]


In [ ]:
# 4. Arange: Like Python's range(), but returns an array
# Use case: Create batch indices, time steps
sequence = np.arange(0, 20, 2)  # start=0, stop=20 (exclusive), step=2
print(f"Arange (0 to 20, step 2): {sequence}")

Arange (0 to 20, step 2): [ 0  2  4  6  8 10 12 14 16 18]


### Create an array of evenly spaced numbers within specified range
```np.linspace(start, stop, num_of_elements, endpoint=True, retstep=False)``` has 5 parameters:
- ```start```: start number (inclusive)
- ```stop```: end number (inclusive unless ```endpoint``` set to ```False```)
- ```num_of_elements```: number of elements contained in the array
- ```endpoint```: boolean value representing whether the ```stop``` number is inclusive or not
- ```retstep```: boolean value representing whether to return the step size

In [ ]:
import numpy as np
arr , step_size = np.linspace(0, 5, 9, endpoint=True, retstep=True)
print(arr)
print('The step size is ' , step_size)

[0.    0.625 1.25  1.875 2.5   3.125 3.75  4.375 5.   ]
The step size is  0.625


In [ ]:
# 5. Linspace: Evenly spaced values (includes endpoint)
# Use case: Generate points for plotting smooth curves
smooth = np.linspace(0, 1, 5)  # 5 values from 0 to 1
print(f"Linspace (5 values from 0 to 1): {smooth}")

# For plotting: 100 points is usually smooth enough
x_plot = np.linspace(0, 2*np.pi, 100)
print(x_plot)
print(f"\nFor plotting sine: {len(x_plot)} points from 0 to 2π")

Linspace (5 values from 0 to 1): [0.   0.25 0.5  0.75 1.  ]
[0.         0.06346652 0.12693304 0.19039955 0.25386607 0.31733259
 0.38079911 0.44426563 0.50773215 0.57119866 0.63466518 0.6981317
 0.76159822 0.82506474 0.88853126 0.95199777 1.01546429 1.07893081
 1.14239733 1.20586385 1.26933037 1.33279688 1.3962634  1.45972992
 1.52319644 1.58666296 1.65012947 1.71359599 1.77706251 1.84052903
 1.90399555 1.96746207 2.03092858 2.0943951  2.15786162 2.22132814
 2.28479466 2.34826118 2.41172769 2.47519421 2.53866073 2.60212725
 2.66559377 2.72906028 2.7925268  2.85599332 2.91945984 2.98292636
 3.04639288 3.10985939 3.17332591 3.23679243 3.30025895 3.36372547
 3.42719199 3.4906585  3.55412502 3.61759154 3.68105806 3.74452458
 3.8079911  3.87145761 3.93492413 3.99839065 4.06185717 4.12532369
 4.1887902  4.25225672 4.31572324 4.37918976 4.44265628 4.5061228
 4.56958931 4.63305583 4.69652235 4.75998887 4.82345539 4.88692191
 4.95038842 5.01385494 5.07732146 5.14078798 5.2042545  5.26772102
 5.3

In [ ]:
# 6. Random arrays: Model initialization, Monte Carlo simulation

# Uniform distribution [0, 1)
uniform = np.random.random((3, 3))
print("Uniform random (3x3):")
print(uniform)

# Normal/Gaussian distribution (mean=0, std=1)
# Use case: Xavier/He initialization for neural networks
normal = np.random.normal(0, 1, (3, 3))
print("\nNormal random (mean=0, std=1):")
print(normal)

Uniform random (3x3):
[[0.65376594 0.07865683 0.0699177 ]
 [0.64437161 0.00672157 0.88821818]
 [0.97799056 0.08439344 0.55089031]]

Normal random (mean=0, std=1):
[[ 0.61471589  0.34769479 -1.14899326]
 [ 0.8627838   0.31779695 -0.47617577]
 [ 0.39239356 -0.17432442 -1.00160283]]


In [ ]:
# 7. Identity matrix: Linear algebra, rotation matrices
eye = np.eye(3)
print("Identity matrix (3x3):")
print(eye)

Identity matrix (3x3):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


### Quick Reference: NumPy Data Types

| Data Type | Description | Common Use Case |
|-----------|-------------|----------------|
| `bool_` | True/False | Masks, conditions |
| `int8` | -128 to 127 | Small counters |
| `int32` | ~±2 billion | Default integers |
| `int64` | Very large ints | Large datasets |
| `float32` | 32-bit float | GPU training (faster) |
| `float64` | 64-bit float | Scientific computing |
| `complex64` | Complex numbers | Signal processing |

---

## Chapter 3: Understanding Array Structure

### When Will You Use This?

- **Debugging**: "Why doesn't my matrix multiplication work?" → Check shapes
- **Memory planning**: "Can I fit this dataset in memory?" → Check size x dtype
- **Broadcasting**: Understanding shapes is KEY for Chapter 8 (Session 2)

### 3.1 The Four Essential Attributes

In [ ]:
import numpy as np
# Create reproducible random arrays using a seed
rng = np.random.default_rng(seed=1701)  # seed ensures same random numbers each run

x1 = rng.integers(10, size=6)           # 1D: 6 elements from 0-9
x2 = rng.integers(100, size=(3, 4))     # 2D: 3x4 matrix
x3 = rng.integers(10, size=(3, 4, 5))   # 3D: 3 matrices of 4x5

print("x1 (1D):", x1)
print("\nx2 (2D):")
print(x2)
print("\nx3 shape:", x3.shape, "(3D - too big to print nicely)")
print(x3)

x1 (1D): [9 4 0 3 8 6]

x2 (2D):
[[39 15 33 79]
 [41  9 25 36]
 [ 1  6 60 96]]

x3 shape: (3, 4, 5) (3D - too big to print nicely)
[[[4 3 5 5 0]
  [8 3 5 2 2]
  [1 8 8 5 3]
  [0 0 8 5 8]]

 [[5 1 6 2 3]
  [1 2 5 6 2]
  [5 2 7 9 3]
  [5 6 0 2 0]]

 [[2 9 4 3 9]
  [9 2 2 4 0]
  [0 3 0 0 2]
  [3 2 7 4 7]]]


In [ ]:
# The 4 essential attributes
print("=" * 40)
print("Array Attributes for x2 (3x4 matrix):")
print("=" * 40)
print(f"ndim:  {x2.ndim}      ← Number of dimensions (axes)")
print(f"shape: {x2.shape}  ← Size along each dimension (rows, cols)")
print(f"size:  {x2.size}     ← Total number of elements (3x4=12)")
print(f"dtype: {x2.dtype}   ← Data type of elements")

Array Attributes for x2 (3x4 matrix):
ndim:  2      ← Number of dimensions (axes)
shape: (3, 4)  ← Size along each dimension (rows, cols)
size:  12     ← Total number of elements (3x4=12)
dtype: int64   ← Data type of elements


### Mental Model: Understanding Dimensions

```
1D Array (shape: (6,))        → A row of 6 boxes
┌─┬─┬─┬─┬─┬─┐
│1│2│3│4│5│6│
└─┴─┴─┴─┴─┴─┘

2D Array (shape: (3, 4))      → 3 rows x 4 columns
┌──┬──┬──┬──┐
│  │  │  │  │  ← Row 0
├──┼──┼──┼──┤
│  │  │  │  │  ← Row 1
├──┼──┼──┼──┤
│  │  │  │  │  ← Row 2
└──┴──┴──┴──┘
 C0 C1 C2 C3

3D Array (shape: (2, 3, 4))   → 2 "pages" of 3x4 matrices
Page 0          Page 1
┌──┬──┬──┬──┐   ┌──┬──┬──┬──┐
│  │  │  │  │   │  │  │  │  │
├──┼──┼──┼──┤   ├──┼──┼──┼──┤
│  │  │  │  │   │  │  │  │  │
├──┼──┼──┼──┤   ├──┼──┼──┼──┤
│  │  │  │  │   │  │  │  │  │
└──┴──┴──┴──┘   └──┴──┴──┴──┘
```

---

## Chapter 4: Accessing Data — Indexing

### When Will You Use This?

- Access specific data points in a dataset
- Get a single prediction from model output
- Extract a specific pixel from an image

### 4.1 One-Dimensional Indexing

In [ ]:
print(f"x1 = {x1}")
print(f"     {''.join([str(i).center(3) for i in range(len(x1))])} ← positive index")
print(f"     {''.join([str(i-len(x1)).center(3) for i in range(len(x1))])} ← negative index")

x1 = [9 4 0 3 8 6]
      0  1  2  3  4  5  ← positive index
      -6 -5 -4 -3 -2 -1 ← negative index


In [ ]:
print(f"x1[0]  = {x1[0]}   ← First element")
print(f"x1[-1] = {x1[-1]}   ← Last element")
print(f"x1[-2] = {x1[-2]}   ← Second to last")

x1[0]  = 9   ← First element
x1[-1] = 6   ← Last element
x1[-2] = 8   ← Second to last


### 4.2 Multi-Dimensional Indexing

**Key Insight**: Use comma-separated indices: `array[row, column]` or `array[page, row, column]`

In [ ]:
print("x2 (3x4 matrix):")
print(x2)
print()
print(f"x2[0, 0] = {x2[0, 0]}  ← Top-left corner (row 0, col 0)")
print(f"x2[1, 0] = {x2[1, 0]}  ← Row 1, col 0")
print(f"x2[2, -1] = {x2[2, -1]}  ← Row 2, last column")

x2 (3x4 matrix):
[[39 15 33 79]
 [41  9 25 36]
 [ 1  6 60 96]]

x2[0, 0] = 39  ← Top-left corner (row 0, col 0)
x2[1, 0] = 41  ← Row 1, col 0
x2[2, -1] = 96  ← Row 2, last column


In [ ]:
# 3D indexing: [page, row, column]
print(f"x3 shape: {x3.shape}")
print(x3)
print(f"x3[1, 2, 3] = {x3[-1, 2, 3]}  ← Page 1, Row 2, Col 3")

x3 shape: (3, 4, 5)
[[[4 3 5 5 0]
  [8 3 5 2 2]
  [1 8 8 5 3]
  [0 0 8 5 8]]

 [[5 1 6 2 3]
  [1 2 5 6 2]
  [5 2 7 9 3]
  [5 6 0 2 0]]

 [[2 9 4 3 9]
  [9 2 2 4 0]
  [0 3 0 0 2]
  [3 2 7 4 7]]]
x3[1, 2, 3] = 0  ← Page 1, Row 2, Col 3


In [ ]:
# 3D indexing: [page, row, column]
print(f"x3 shape: {x3.shape}")
print(x3)
print(f"x3[1, 2, 3] = {x3[1, 2, 3]}  ← Page 1, Row 2, Col 3")

x3 shape: (3, 4, 5)
[[[4 3 5 5 0]
  [8 3 5 2 2]
  [1 8 8 5 3]
  [0 0 8 5 8]]

 [[5 1 6 2 3]
  [1 2 5 6 2]
  [5 2 7 9 3]
  [5 6 0 2 0]]

 [[2 9 4 3 9]
  [9 2 2 4 0]
  [0 3 0 0 2]
  [3 2 7 4 7]]]
x3[1, 2, 3] = 9  ← Page 1, Row 2, Col 3


### 4.3 Modifying Values

In [ ]:
print("Before modification:")
print(x2)

x2[0, 0] = 999  # Modify single element

print("\nAfter x2[0, 0] = 999:")
print(x2)

Before modification:
[[39 15 33 79]
 [41  9 25 36]
 [ 1  6 60 96]]

After x2[0, 0] = 999:
[[999  15  33  79]
 [ 41   9  25  36]
 [  1   6  60  96]]


---

## Chapter 5: Accessing Data — Slicing

### When Will You Use This?

| Task | Slicing Example |
|------|----------------|
| Get first 100 samples | `data[:100]` |
| Split train/test | `train = data[:800]`, `test = data[800:]` |
| Select every 5th frame | `video[::5]` |
| Reverse time series | `prices[::-1]` |
| Get all rows, specific columns | `data[:, [0, 2, 5]]` |

### 5.1 The Slice Syntax: `start:stop:step`

```
array[start:stop:step]
       │     │    └── Every nth element (default: 1)
       │     └─────── Stop BEFORE this index (exclusive)
       └───────────── Start AT this index (inclusive)
```

In [ ]:
x = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
print(f"x = {x}")
print(f"    {' '.join([str(i) for i in range(10)])}  ← indices")

x = [0 1 2 3 4 5 6 7 8 9]
    0 1 2 3 4 5 6 7 8 9  ← indices


In [ ]:
print(f"x[2:7]   = {x[2:7]}      ← From index 2 to 6 (stop is exclusive!)")
print(f"x[:5]    = {x[:5]}         ← First 5 elements")
print(f"x[5:]    = {x[5:]}         ← From index 5 to end")
print(f"x[::2]   = {x[::2]}      ← Every 2nd element (indices 0,2,4,6,8)")
print(f"x[1::2]  = {x[1::2]}      ← Every 2nd, starting at index 1")
print(f"x[::-1]  = {x[::-1]}  ← Reversed!")

x[2:7]   = [2 3 4 5 6]      ← From index 2 to 6 (stop is exclusive!)
x[:5]    = [0 1 2 3 4]         ← First 5 elements
x[5:]    = [5 6 7 8 9]         ← From index 5 to end
x[::2]   = [0 2 4 6 8]      ← Every 2nd element (indices 0,2,4,6,8)
x[1::2]  = [1 3 5 7 9]      ← Every 2nd, starting at index 1
x[::-1]  = [9 8 7 6 5 4 3 2 1 0]  ← Reversed!


### 5.2 Multi-Dimensional Slicing

In [ ]:
# Reset x2 for clean examples
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8],
               [9, 10, 11, 12]])
print("x2:")
print(x2)

x2:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


In [ ]:
# Combine row and column slices
print("x2[:2, :3]  ← First 2 rows, first 3 columns:")
print(x2[:2, :3])

x2[:2, :3]  ← First 2 rows, first 3 columns:
[[1 2 3]
 [5 6 7]]


In [ ]:
print("x2[:, ::2]  ← All rows, every 2nd column:")
print(x2[:, ::2])

x2[:, ::2]  ← All rows, every 2nd column:
[[ 1  3]
 [ 5  7]
 [ 9 11]]


In [ ]:
print("x2[::-1, ::-1]  ← Reversed both dimensions (180° rotation):")
print(x2[::-1, ::-1])

x2[::-1, ::-1]  ← Reversed both dimensions (180° rotation):
[[12 11 10  9]
 [ 8  7  6  5]
 [ 4  3  2  1]]


### 5.3 Accessing Rows and Columns

In [ ]:
print("Original x2:")
print(x2)
print()
print(f"x2[1, :]  = {x2[1, :]}  ← Row 1 (all columns)")
print(f"x2[1]     = {x2[1]}  ← Same! Row 1 (shorthand)")
print(f"x2[:, 2]  = {x2[:, 2]}    ← Column 2 (all rows)")

Original x2:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

x2[1, :]  = [5 6 7 8]  ← Row 1 (all columns)
x2[1]     = [5 6 7 8]  ← Same! Row 1 (shorthand)
x2[:, 2]  = [ 3  7 11]    ← Column 2 (all rows)


---

##Chapter 6: Views vs. Copies — The #1 Source of NumPy Bugs!

###This Will Save You Hours of Debugging!

**The Trap**: When you slice a NumPy array, you get a **VIEW**, not a copy. Modifying the slice modifies the original!

In [ ]:
# The dangerous pattern that catches everyone
original = np.array([[1, 2, 3],
                     [4, 5, 6],
                     [7, 8, 9]])
print("Original:")
print(original)

# This creates a VIEW, not a copy!
subset = original[:2, :2]
print("\nSubset (top-left 2x2):")
print(subset)

Original:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Subset (top-left 2x2):
[[1 2]
 [4 5]]


In [ ]:
# Now modify the subset...
subset[0, 0] = 999
print("After setting subset[0,0] = 999:")
print("\nSubset:")
print(subset)
print("\n ORIGINAL (also changed!):")
print(original)

After setting subset[0,0] = 999:

Subset:
[[999   2]
 [  4   5]]

 ORIGINAL (also changed!):
[[999   2   3]
 [  4   5   6]
 [  7   8   9]]


### The Solution: Use `.copy()`

In [ ]:
# Reset original
original = np.array([[1, 2, 3],
                     [4, 5, 6],
                     [7, 8, 9]])

# Create an ACTUAL copy
safe_subset = original[:2, :2].copy()  # ← THE FIX!

safe_subset[0, 0] = 999
print("After modifying the COPY:")
print("\nSafe subset:")
print(safe_subset)
print("\n Original (unchanged!):")
print(original)

After modifying the COPY:

Safe subset:
[[999   2]
 [  4   5]]

 Original (unchanged!):
[[1 2 3]
 [4 5 6]
 [7 8 9]]


### View vs. Copy Cheat Sheet

| Operation | Returns | Example |
|-----------|---------|--------|
| Basic slicing | **VIEW** | `arr[0:5]` |
| Reshape | **VIEW** | `arr.reshape(2, 3)` |
| Transpose | **VIEW** | `arr.T` |
| `.copy()` | **COPY** | `arr[0:5].copy()` |
| Fancy indexing | **COPY** | `arr[[0, 2, 4]]` |

**Rule of thumb**: If you're going to modify a subset and don't want to affect the original, always use `.copy()`!

---

## Chapter 7: Reshaping and Restructuring Arrays

### When Will You Use This?

| Task | Method |
|------|--------|
| Flatten image for neural network input | `image.reshape(-1)` |
| Convert 1D predictions to 2D grid | `predictions.reshape(10, 10)` |
| Add batch dimension | `data[np.newaxis, :]` |
| Stack multiple arrays | `np.vstack()`, `np.hstack()` |

### 7.1 Reshape

In [ ]:
# 1D to 2D
arr = np.arange(1, 10)
print(f"Original: {arr}, shape: {arr.shape}")

grid = arr.reshape(3, 3)
print(f"\nReshaped to 3x3:")
print(grid)

Original: [1 2 3 4 5 6 7 8 9], shape: (9,)

Reshaped to 3x3:
[[1 2 3]
 [4 5 6]
 [7 8 9]]


In [ ]:
# Size must match!
try:
    bad = arr.reshape(3, 4)  # 9 elements can't fit in 3x4=12 spots
except ValueError as e:
    print(f"Error: {e}")

Error: cannot reshape array of size 9 into shape (3,4)


In [ ]:
# Reshape returns a VIEW!
grid[0, 0] = 999
print(f"After modifying grid[0,0]:")
print(f"Grid: \n{grid}")
print(f"Original arr: {arr}  ← Also changed!")

After modifying grid[0,0]:
Grid: 
[[999   2   3]
 [  4   5   6]
 [  7   8   9]]
Original arr: [999   2   3   4   5   6   7   8   9]  ← Also changed!


### 7.2 Adding Dimensions with `np.newaxis`

In [ ]:
x = np.array([1, 2, 3])
print(f"Original: {x}, shape: {x.shape}")

# Convert to row vector (1x3)
row = x[np.newaxis, :]
print(f"\nRow vector: {row}, shape: {row.shape}")

# Convert to column vector (3x1)
col = x[:, np.newaxis]
print(f"\nColumn vector:\n{col}\nshape: {col.shape}")

Original: [1 2 3], shape: (3,)

Row vector: [[1 2 3]], shape: (1, 3)

Column vector:
[[1]
 [2]
 [3]]
shape: (3, 1)


### 7.3 Concatenation: Joining Arrays

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# 1D concatenation
print(f"np.concatenate([a, b]): {np.concatenate([a, b])}")

np.concatenate([a, b]): [1 2 3 4 5 6]


In [ ]:
# 2D concatenation
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])

print("Original grid:")
print(grid)

print("\nnp.concatenate([grid, grid], axis=0)  ← Stack vertically:")
print(np.concatenate([grid, grid], axis=0))

print("\nnp.concatenate([grid, grid], axis=1)  ← Stack horizontally:")
print(np.concatenate([grid, grid], axis=1))

Original grid:
[[1 2 3]
 [4 5 6]]

np.concatenate([grid, grid], axis=0)  ← Stack vertically:
[[1 2 3]
 [4 5 6]
 [1 2 3]
 [4 5 6]]

np.concatenate([grid, grid], axis=1)  ← Stack horizontally:
[[1 2 3 1 2 3]
 [4 5 6 4 5 6]]


### 7.4 `vstack` and `hstack`: Easier Stacking

In [ ]:
x = np.array([1, 2, 3])
grid = np.array([[7, 8, 9],
                 [10, 11, 12]])

# vstack: Stack vertically (add rows)
print("np.vstack([x, grid]):")
print(np.vstack([x, grid]))

np.vstack([x, grid]):
[[ 1  2  3]
 [ 7  8  9]
 [10 11 12]]


In [ ]:
y = np.array([[99], [99]])

# hstack: Stack horizontally (add columns)
print("np.hstack([grid, y]):")
print(np.hstack([grid, y]))

np.hstack([grid, y]):
[[ 7  8  9 99]
 [10 11 12 99]]


### 7.5 Splitting Arrays

In [ ]:
x = np.array([1, 2, 3, 99, 99, 3, 2, 1])

# Split at indices 3 and 5
x1, x2, x3 = np.split(x, [3, 5])
print(f"Original: {x}")
print(f"Split at [3, 5]:")
print(f"  x1: {x1}")
print(f"  x2: {x2}")
print(f"  x3: {x3}")

Original: [ 1  2  3 99 99  3  2  1]
Split at [3, 5]:
  x1: [1 2 3]
  x2: [99 99]
  x3: [3 2 1]


In [ ]:
grid = np.arange(16).reshape(4, 4)
print("Grid (4x4):")
print(grid)

# vsplit: Split vertically (by rows)
upper, lower = np.vsplit(grid, [2])
print("\nnp.vsplit(grid, [2]):")
print(f"Upper:\n{upper}")
print(f"Lower:\n{lower}")

Grid (4x4):
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]

np.vsplit(grid, [2]):
Upper:
[[0 1 2 3]
 [4 5 6 7]]
Lower:
[[ 8  9 10 11]
 [12 13 14 15]]


In [ ]:
# hsplit: Split horizontally (by columns)
left, right = np.hsplit(grid, [2])
print("np.hsplit(grid, [2]):")
print(f"Left:\n{left}")
print(f"Right:\n{right}")

np.hsplit(grid, [2]):
Left:
[[ 0  1]
 [ 4  5]
 [ 8  9]
 [12 13]]
Right:
[[ 2  3]
 [ 6  7]
 [10 11]
 [14 15]]


---

## Session 1 Summary

### What You Learned

| Chapter | Key Concept | Data Science Application |
|---------|-------------|-------------------------|
| 1 | NumPy is 100x faster | Handle million-row datasets |
| 2 | Array creation | Initialize models, load data |
| 3 | Shape & dtype | Debug dimension mismatches |
| 4-5 | Indexing & slicing | Filter and subset data |
| 6 | Views vs. copies | Avoid data corruption bugs |
| 7 | Reshape & stack | Prepare data for ML models |

### Coming in Session 2

- **Chapter 8: Aggregations** (sum, mean, min, max) — The start of EDA
- **Chapter 9: Broadcasting** — Magic for array arithmetic
- **Chapter 10: Boolean Masking** — Filter data like a pro

---

## Practice Exercises

Try these before Session 2!

In [2]:
# Exercise 1: Create a 5x5 identity matrix, then set all corner values to 9
# Your code here:
import numpy as np
id=np.eye(5)
id[0,0]=9
id[0,4]=9
id[4,0]=9
id[4,4]=9
print(id)

[[9. 0. 0. 0. 9.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [9. 0. 0. 0. 9.]]


In [6]:
# Exercise 2: Create an array of 100 evenly spaced values from 0 to 2π
# Then reshape it to a 10x10 matrix
# Your code here:
arr=np.linspace(0,2*np.pi,100)
print(arr.reshape(10,10))

[[0.         0.06346652 0.12693304 0.19039955 0.25386607 0.31733259
  0.38079911 0.44426563 0.50773215 0.57119866]
 [0.63466518 0.6981317  0.76159822 0.82506474 0.88853126 0.95199777
  1.01546429 1.07893081 1.14239733 1.20586385]
 [1.26933037 1.33279688 1.3962634  1.45972992 1.52319644 1.58666296
  1.65012947 1.71359599 1.77706251 1.84052903]
 [1.90399555 1.96746207 2.03092858 2.0943951  2.15786162 2.22132814
  2.28479466 2.34826118 2.41172769 2.47519421]
 [2.53866073 2.60212725 2.66559377 2.72906028 2.7925268  2.85599332
  2.91945984 2.98292636 3.04639288 3.10985939]
 [3.17332591 3.23679243 3.30025895 3.36372547 3.42719199 3.4906585
  3.55412502 3.61759154 3.68105806 3.74452458]
 [3.8079911  3.87145761 3.93492413 3.99839065 4.06185717 4.12532369
  4.1887902  4.25225672 4.31572324 4.37918976]
 [4.44265628 4.5061228  4.56958931 4.63305583 4.69652235 4.75998887
  4.82345539 4.88692191 4.95038842 5.01385494]
 [5.07732146 5.14078798 5.2042545  5.26772102 5.33118753 5.39465405
  5.45812057 

In [7]:
# Exercise 3: Given this array, extract every other element in reverse order
arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
# Expected output: [10, 8, 6, 4, 2]
# Your code here:
arr[::-2]

array([10,  8,  6,  4,  2])

In [12]:
# Exercise 4: Create a 4x4 matrix, then safely extract the top-right 2x2 corner
# (Make sure modifying your extraction doesn't affect the original!)
# Your code here:
original=[[1,2,3,9],
          [4,5,6,9],
          [7,8,9,9]]
subset=original[:2,2:].copy()
print(subset)

TypeError: list indices must be integers or slices, not tuple